# Persistent HMM experiments: Gaussian, GMM, and two spline-flow emissions

Compare **four emissions** on the **exact saved observations** used by the existing
Gaussian/GMM experiments: Gaussian, three-component GMM, RQ-NSF with a Gaussian base,
and RQ-NSF with a Student-t base.

Both **mean-separated** and **volatility-only** regimes use three states with
self-transition probability **0.995**. The saved runs contain four generating
families: Gaussian, Student-t, trimodal, and skewed Gamma.

**The five-iteration patience version has not been run.** Completed tuning
selections are reused; final fits are saved under a new run name.
Both flow variants use hmmlearn compiled inference and Adam emission updates.
`EMISSION_BATCH_SIZE` controls the sampled observations per gradient update;
`None` retains full-batch training. The E-step still uses the complete sequence.
Mini-batch updates are accepted only when their final full-data weighted objective
improves on the starting parameters. Mini-batch losses alone do not decide this.

Restart the kernel before running this version to load the new implementation.
The separate run name preserves earlier results. Execution switches retain your
selected values; running all cells with them enabled launches the full run.

CPU toy benchmarks favored independent worker processes over extra threads within
one fit. `N_JOBS` controls workers; `TORCH_THREADS` limits threads within each.
The default is eight CPU workers with one thread each. Tuning parallelizes
independent validation folds; final fitting parallelizes independent restarts.
One worker pool is reused throughout each stage. `N_JOBS=1` runs sequentially.
The new run name preserves earlier outputs; restart the kernel before running.
`CONVERGENCE_PATIENCE=5` requires five consecutive absolute full-data
log-likelihood changes below tolerance, measured after each M-step. Reaching the
iteration cap alone is not reported as tolerance convergence. Stop an older run
and restart the kernel before running this version. `RUN_TUNING=False` reuses
the completed sweep; no new sweep is needed to run these final fits.


## Settings and recovered source runs

These are the saved **0.995** runs, rather than the older 0.9-persistence runs.
The mean-separated run uses seed 42 and means $(-2,0,2)$ with variance 1. The
volatility-only run uses seed 31, zero means, and standard deviations $(1,2,4)$.
Each has 10 replicates, 20,000 training observations, 5,000 independent test
observations, 10 starts, and a 1,000-update maximum. Pair models **within** each
regime/family/replicate; the mean and volatility datasets are different experiments.

The helper restores saved arrays and selected baseline parameters; it does not
regenerate data or retrain Gaussian/GMM baselines. Replayed state metrics must
match saved metrics to absolute tolerance $10^{-8}$ before inclusion.

In [ ]:
from dataclasses import asdict
from pathlib import Path
import itertools
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

project_root = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src/flexible_emission_hmm").is_dir()
)
for path in (project_root / "src", project_root / "notebooks"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import nf_hmmlearn_comparison as experiment

plt.rcParams.update({"font.size": 10, "axes.grid": False})
pd.set_option("display.max_columns", 20)

SOURCES = dict(experiment.SOURCES)
DEVICE = "cpu" 
DTYPE = "float64"
N_JOBS = 8  # Independent CPU worker processes. Set 1 for sequential fits.
TORCH_THREADS = 1  # PyTorch/BLAS threads per worker.
EMISSION_BATCH_SIZE = 2048  # None uses full-data gradient updates.
CONVERGENCE_PATIENCE = 5  # Consecutive small post-update likelihood changes.
BASE_FLOW = experiment.FlowSettings(
    bound=5.0, base_df=5.0, m_step_steps=10, batch_size=EMISSION_BATCH_SIZE,
    device=DEVICE, dtype=DTYPE, convergence_patience=CONVERGENCE_PATIENCE
)
TUNING = experiment.TuningSettings(
    count_bins=(4, 8, 16),
    num_layers=(1, 2),
    m_step_steps=(10, 20),
    learning_rates=(1e-3,),
    student_dfs=(5.0,),  # Fixed Student-t degrees of freedom.
    replicates=(0, 1, 2),
    train_fraction=0.8,
    n_starts=3,  # Tuning only; final fits inherit 10 starts.
    max_iter=150,
)

# Reuse the completed sweep. To retune, choose a new TUNING_RUN_NAME first.
TUNING_RUN_NAME = (
    "four_emissions_p995_hmmlearn_m10_l2_b8-16_s2_i150"
    "_batch2048_float64_cpu8x1_v1"
)
RUN_NAME = f"{TUNING_RUN_NAME}_patience{CONVERGENCE_PATIENCE}_v2"
experiment_dir = project_root / "data/synthetic/nf_spline_experiments"
tuning_dir = experiment_dir / TUNING_RUN_NAME / "tuning"
run_dir = experiment_dir / RUN_NAME
RUN_TUNING = False  # The completed sweep already selected the hyperparameters.
RUN_FINAL = True
REPORT_EXISTING = False

sources = experiment.load_sources(project_root, SOURCES)
display(pd.DataFrame([
    dict(regime=regime, source=str(source.path.relative_to(project_root)),
         persistence=source.transitions[0, 0], **source.settings)
    for regime, source in sources.items()
]).drop(columns=["quad_epsabs", "quad_epsrel", "quad_limit"], errors="ignore"))
print(f"Emission device: {DEVICE}; dtype: {DTYPE}")
print("Flow backend: hmmlearn; tuning M-step steps:", TUNING.m_step_steps)
print("Emission batch size:", EMISSION_BATCH_SIZE or "full dataset")
print("Convergence patience:", CONVERGENCE_PATIENCE)
print("Saved tuning:", tuning_dir)
print("Final output:", run_dir)
print(f"CPU workers: {N_JOBS}; threads per worker: {TORCH_THREADS}")
print("Tuning enabled:", RUN_TUNING, "Final training enabled:", RUN_FINAL)

## Model and metric definitions

All models use the same HMM equations and expected-count transition updates.
The two flow models now use `hmmlearn_flow_hmm.FlowHMM` with compiled CPU
forward–backward, filtering, and Viterbi. Their chain initialization remains uniform
to match the previous experiment; Gaussian/GMM baselines are still restored.
Flows use generalized Baum–Welch with gradient-based emission M-steps. For a flow
mapping observations to base variables,

$$
\log p_k(x)=\log p_{\mathrm{base}}(f_k(x))+\log|f_k'(x)|.
$$

The normal base has unit variance. The Student-t base is also standardized to unit
variance using scale $\sqrt{(\nu-2)/\nu}$; $\nu>2$ ensures finite emission variance.
Its degrees of freedom are fixed at **5** for tuning and final fitting. The spline
bound is fixed at **5**, so its normalized transformation interval is $[-5,5]$.
Identity spline tails retain Gaussian or Student-t tail
behavior, respectively. Both variants retain the same trainable location/scale.

| Metric group | Metrics retained from the reference experiments |
|---|---|
| State classification | Accuracy; macro/weighted precision, recall, F1; balanced accuracy; MCC; ARI; NMI; per-state reports and confusion matrices |
| State probabilities | Capped state log loss and cap/zero fractions; Brier score; mean true-state posterior |
| Switching | Switch precision, recall, F1; rates; signed and absolute switch-count errors |
| Density and parameter recovery | Test NLL per observation; clean-target emission KL; mean RMSE; variance RMSE; quadrature errors |
| Reliability and cost | Start failures; convergence; iterations; likelihood decreases; selected and total fit times |

State labels are aligned by Hungarian assignment on training Viterbi states and
then frozen for every test condition. Test labels never select a start or setting.
Flow KL and fitted moments use adaptive **1D quadrature**, with errors/warnings
exported. Diagnostic evaluation uses CPU float64 copies of learned parameters;
training and test likelihoods use the configured device/dtype. The original
Gaussian/GMM density-recovery metrics are retained from their saved runs.

## Training-only hyperparameter tuning

For each regime, generating family, and flow base, fit on the first 80% of each
configured training replicate. Choose each fold's restart by prefix training
likelihood, then score its remaining 20% using conditional predictive NLL. The
validation prior is the prefix's final filtered distribution propagated through
the fitted transition matrix. Do not concatenate folds.

`TUNING` defines the bin/layer grids, Adam updates per M-step, validation
replicates, restart count, and outer-iteration cap. `BASE_FLOW.batch_size` is fixed
across candidates and inherited by both bases and final fits. A batch size of
2,048 draws that many observations uniformly with replacement per gradient
update, using their fixed E-step weights. `m_step_steps` counts sampled-batch
updates per state, not complete passes through the data. None or a batch size at
least as large as the fitted dataset uses the full-batch method.

Rank settings by mean validation NLL across replicate suffixes. A candidate is
eligible only if all folds succeed; ties follow grid order. Tune the two flow
bases independently. Bound and Student-t degrees of freedom remain fixed at 5.
Final fitting refits on each full saved training sequence with ten starts and a
1,000-update cap. A finite search does not guarantee optimal hyperparameters.

The notebook displays fit counts before starting. Completed starts and groups
are checkpointed. Changing settings or source hashes requires a new run name.

Parallel execution preserves the same seeds, grid order, fold eligibility, and
training-likelihood restart selection. Workers save distinct fold/start
checkpoints; the parent writes combined tuning tables and selected settings.
The request records worker and thread counts. GPU settings require `N_JOBS=1`.
This version reuses the completed sweep's selections. Those validation scores
were obtained with the earlier stopping rule. New tuning, if desired, uses
`BASE_FLOW.convergence_patience`; set a new `TUNING_RUN_NAME` before enabling it.


In [ ]:
common_candidates = (len(TUNING.count_bins) * len(TUNING.num_layers)
                     * len(TUNING.m_step_steps) * len(TUNING.learning_rates))
family_groups = sum(len(source.families) for source in sources.values())
tuning_fits = (family_groups * common_candidates * (1 + len(TUNING.student_dfs))
               * len(TUNING.replicates) * TUNING.n_starts)
final_fits = sum(
    len(source.families) * len(source.settings["train_sizes"])
    * source.settings["n_replicates"] * source.settings["n_starts"] * 2
    for source in sources.values()
)
display(pd.DataFrame([
    {"stage": "Training-only tuning", "maximum fitting attempts": tuning_fits},
    {"stage": "Final fits: both flow bases", "maximum fitting attempts": final_fits},
    {"stage": "Gaussian/GMM refits", "maximum fitting attempts": 0},
]))

if RUN_TUNING:
    torch.set_num_threads(TORCH_THREADS)
    selections, tuning_rows = experiment.tune_flows(
        sources, tuning_dir, BASE_FLOW, TUNING,
        n_jobs=N_JOBS, torch_threads=TORCH_THREADS,
    )
else:
    print("Tuning disabled; no hyperparameters have been selected by this cell.")

### Review validation scores and selected settings

The first table shows the winning hyperparameters and **minimum mean validation
NLL** for each regime, data family, and flow base. Lower NLL is better within
each group. These scores average the tuning replicates; they are not final test
scores. The second table includes all candidates and their fold coverage.

This cell can run independently after a kernel restart. It uses the configured
`tuning_dir` when available; otherwise it opens the saved CPU tuning run named in
the cell. It only reads results and does not launch training.


In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

# Read the completed sweep independently after restarting the kernel.
if "tuning_dir" not in globals():
    project_root = next(
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "src/flexible_emission_hmm").is_dir()
    )
    review_run_name = globals().get("TUNING_RUN_NAME", (
        "four_emissions_p995_hmmlearn_m10_l2_b8-16_s2_i150"
        "_batch2048_float64_cpu8x1_v1"
    ))
    tuning_dir = (
        project_root / "data/synthetic/nf_spline_experiments"
        / review_run_name / "tuning"
    )

print(f"Reviewing saved tuning: {tuning_dir.parent.name}")
selection_path = tuning_dir / "selected_settings.json"
tuning_path = tuning_dir / "tuning_results.json"
if selection_path.exists() and tuning_path.exists():
    selections = json.loads(selection_path.read_text())
    tuning_rows = json.loads(tuning_path.read_text())
    tuning_table = pd.json_normalize(tuning_rows)
    selection_table = pd.json_normalize(selections)
    winning_settings = selection_table.rename(columns={
        "regime": "Regime",
        "family": "Data family",
        "settings.base_distribution": "Flow base",
        "settings.count_bins": "Bins",
        "settings.num_layers": "Layers",
        "settings.m_step_steps": "M-step updates",
        "validation_nll": "Minimum validation NLL",
    })[[
        "Regime", "Data family", "Flow base", "Bins", "Layers",
        "M-step updates", "Minimum validation NLL",
    ]].sort_values(["Regime", "Data family", "Flow base"])
    print("Selected hyperparameters: lowest mean validation NLL in each group")
    display(winning_settings.style.format({
        "Minimum validation NLL": "{:.6f}",
    }).hide(axis="index"))
    validation_summary = tuning_table.groupby([
        "regime", "family", "settings.base_distribution", "candidate"
    ]).agg(folds=("status", "size"),
           successful=("status", lambda s: s.eq("ok").sum()),
           validation_nll=("validation_nll", "mean"))
    print("All candidates and validation fold coverage:")
    display(validation_summary)
else:
    print("No saved tuning results found. Complete tuning before reviewing them.")

Reviewing saved tuning: four_emissions_p995_hmmlearn_m10_l2_b8-16_s2_i150_batch2048_float64_cpu8x1_v1
Selected hyperparameters: lowest mean validation NLL in each group


Regime,Data family,Flow base,Bins,Layers,M-step updates,Minimum validation NLL
mean,Gaussian,normal,4,1,20,1.463298
mean,Gaussian,student_t,4,1,20,1.470047
mean,Skewed Gamma,normal,8,2,20,1.370706
mean,Skewed Gamma,student_t,8,2,20,1.375350
mean,Student-t,normal,8,1,20,1.100618
mean,Student-t,student_t,4,2,20,1.078918
mean,Trimodal,normal,8,2,20,1.184094
mean,Trimodal,student_t,8,2,20,1.192532
volatility,Gaussian,normal,4,1,20,2.201107
volatility,Gaussian,student_t,4,1,20,2.187306


All candidates and validation fold coverage:


folds  successful  \
regime     family   settings.base_distribution candidate                      
mean       Gaussian normal                     0              3           3   
                                               1              3           3   
                                               2              3           3   
                                               3              3           3   
                                               4              3           3   
...                                                         ...         ...   
volatility Trimodal student_t                  7              3           3   
                                               8              3           3   
                                               9              3           3   
                                               10             3           3   
                                               11             3           3   

                                                          validation_nll  
regime     family   settings.base_distribution candidate                  
mean       Gaussian normal                     0                1.464174  
                                               1                1.463298  
                                               2                1.464829  
                                               3                1.464144  
                                               4                1.465043  
...                                                                  ...  
volatility Trimodal student_t                  7                2.070041  
                                               8                2.065606  
                                               9                2.023411  
                                               10               2.019464  
                                               11               2.019893  

[192 rows x 3 columns]

## Final four-model experiments on recovered datasets

This stage preserves the source runs' sizes, replicates, starts, iteration caps,
and likelihood tolerances. No new training/test observations are generated.
Noise strengths are 0, 10%, 25%, and 50% of pooled training standard deviation,
ramping linearly across the saved test path, with the original deterministic seeds.
Models are trained once on clean data and reused for every noise condition.

For each group, save recovered data, noise realizations, predictions, posteriors,
flow checkpoints, fitting histories, metrics, alignment, and baseline replay checks.
The top-level manifest records input file hashes, settings, software versions, and
code hashes. Existing source artifacts are never overwritten.

The flows use compiled hmmlearn inference with patient stopping: require
`CONVERGENCE_PATIENCE` consecutive absolute full-data log-likelihood changes at
most the tolerance, measured after each M-step. A larger change resets the count.
Histories include the initial score and every actual post-update score. Restart
selection uses the final training score. `converged` requires all five small
changes; `iteration_limit_reached`, the trailing small-change count, and
`stop_reason` are saved separately. The final cap remains 1,000 M-steps.

The completed sweep supplies the selected hyperparameters. Final fits use a new
output directory, so old early-stopped checkpoints are not resumed or overwritten.
The saved validation scores retain their original stopping rule.

Final fits inherit the selected emission batch size and M-step update count.
Batch sampling uses a local seeded generator; checkpoints preserve its state.
The full sequence remains in memory for E-step inference and acceptance checks;
mini-batching reduces gradient work, not the storage of all observations.

In [ ]:
if RUN_FINAL:
    if not selection_path.exists():
        raise RuntimeError("Run training-only tuning before enabling final fitting.")
    torch.set_num_threads(TORCH_THREADS)
    # Apply patient stopping to the existing sweep's chosen hyperparameters.
    selections = [
        {**row, "settings": {
            **row["settings"], "convergence_patience": CONVERGENCE_PATIENCE,
        }}
        for row in experiment.read_json(selection_path)
    ]
    run = experiment.run_comparison(
        sources, run_dir, selections, n_jobs=N_JOBS, torch_threads=TORCH_THREADS
    )
else:
    print("Final fitting disabled; no models trained or evaluated.")

report_ready = RUN_FINAL or REPORT_EXISTING
if REPORT_EXISTING and not RUN_FINAL:
    names = ["results", "attempts", "per_state", "confusion", "baseline_replay",
             "parameters", "coverage"]
    missing = [name for name in names if not (run_dir / f"{name}.csv").exists()]
    if missing:
        raise RuntimeError(f"Missing output tables: {missing}")
    run = {name: pd.read_csv(run_dir / f"{name}.csv") for name in names}


def show_and_save(fig, name):
    """Save a report figure beside the run tables and display it."""
    fig.savefig(run_dir / f"{name}.png", dpi=160, bbox_inches="tight")
    plt.show()

## Coverage and exact baseline replay

Main comparisons require **all four models** to succeed on the same replicate.
Clean comparisons and all-condition robustness comparisons report their own
coverage. A failure must remain visible, not silently reduce only one model's
sample. Density plots additionally require finite values for every model.
Historical baseline fitting times describe earlier runs, not matched hardware
benchmarks against the new flow fits.

In [ ]:
if report_ready:
    results = run["results"]
    clean = experiment.complete_groups(results, ["clean"])
    robust = experiment.complete_groups(results)
    coverage = results.assign(ok=results.evaluation_status.eq("ok")).groupby([
        "regime", "family", "model", "condition"
    ]).agg(evaluations=("ok", "size"), successful=("ok", "sum"))
    display(coverage)
    display(run["coverage"])
    replay = run["baseline_replay"]
    if replay.empty or not replay.passed.all():
        raise RuntimeError("Some baseline replays failed; inspect baseline_replay.csv.")
    print("Saved baseline metric replay checks:", len(replay))
    display(clean.groupby(["regime", "family", "model"]).replicate.nunique())
    errors = results[results.evaluation_status != "ok"]
    if not errors.empty:
        display(errors[["regime", "family", "replicate", "model", "condition", "error"]])
    numeric_metrics = [m for m in experiment.METRICS if m in results]
    summary = clean.groupby(["regime", "family", "n_train", "model"])[
        numeric_metrics
    ].agg(["mean", "std", "count"])
    summary.to_csv(run_dir / "clean_metric_summary.csv")
    display(summary)
else:
    print("No report computed: experiments have not been run.")

## Classification learning curves

Match the reference accuracy, macro precision, macro recall, and macro F1 panels.
Each panel shows replicate mean ± one sample SD, not a confidence interval.
The recovered runs have one training size (20,000), so these are single-point
comparisons; extra learning-curve sizes require fitting new matched baselines.

In [ ]:
if report_ready and not clean.empty:
    for regime in sources:
        data = experiment.metric_groups(clean, [
            "aligned_accuracy", "precision_macro", "recall_macro", "f1_macro"
        ])
        if not data[data.regime == regime].empty:
            fig = experiment.plot_metrics(data, regime, [
                "aligned_accuracy", "precision_macro", "recall_macro", "f1_macro"
            ])
            show_and_save(fig, f"{regime}_classification_learning_curves")

## Density, probability, transition, and parameter recovery

Test NLL is the fitted model's sequence NLL per observation, compared only within
the same regime/family. Emission KL is $\mathrm{KL}(p_{\rm true}\|p_{\rm fitted})$
averaged equally across aligned states and targets the **clean** generating density.
It stays unchanged when only test measurement noise changes. Quadrature errors and
warnings must be inspected alongside recovery metrics.

In [ ]:
if report_ready and not clean.empty:
    for regime in sources:
        for label, metrics in [
            ("recovery", ["test_nll", "emission_kl", "state_brier_score", "transition_rmse"]),
            ("parameters", ["mean_rmse", "variance_rmse", "state_log_loss", "switch_f1"]),
        ]:
            data = experiment.metric_groups(clean, metrics)
            if not data[data.regime == regime].empty:
                show_and_save(experiment.plot_metrics(data, regime, metrics),
                              f"{regime}_{label}_learning_curves")
    diagnostic_columns = [c for c in [
        "regime", "family", "replicate", "model", "recovery_status",
        "kl_quad_error_max", "moment_quad_error_max", "evaluation_warnings"
    ] if c in results]
    display(results[results.condition == "clean"][diagnostic_columns])

## All six paired model differences

Subtract within a replicate before averaging. Intervals are 95% percentile
bootstrap intervals for the mean difference, resampling independent replicates
2,000 times. All four models must have finite values for the metric; never
bootstrap individual time points. Positive F1 differences favor the second model;
negative NLL/KL differences favor the second model. Timing comparisons remain
historical and should not be interpreted as controlled speed measurements.

Difference-plot abbreviations: NSF-N = Gaussian-base spline flow; NSF-t = Student-t-base spline flow.

Intervals are conditional on the selected hyperparameters; the bootstrap does not rerun tuning or include tuning-selection uncertainty.

In [ ]:
if report_ready and not clean.empty:
    delta_metrics = ["f1_macro", "test_nll", "emission_kl", "total_fit_seconds"]
    deltas = experiment.paired_differences(clean, delta_metrics)
    deltas.to_csv(run_dir / "paired_model_differences.csv", index=False)
    display(deltas)
    for regime in sources:
        if not deltas[deltas.regime == regime].empty:
            show_and_save(experiment.plot_differences(deltas, regime, delta_metrics),
                          f"{regime}_paired_model_differences")

## Confusion matrices and per-state classification

Each matrix pools the same complete four-model replicate groups. Rows are true
states; columns are aligned predicted states. Tiles show counts and row percentages
on a common 0–100% color scale. Per-state precision/recall/F1 use the same groups.

In [ ]:
if report_ready and not clean.empty:
    keys = experiment.KEYS + ["model", "condition"]
    state_rows = run["per_state"].merge(clean[keys], on=keys, validate="many_to_one")
    state_summary = state_rows.groupby([
        "regime", "family", "n_train", "model", "state"
    ])[["precision", "recall", "f1", "support"]].agg(["mean", "std"])
    state_summary.to_csv(run_dir / "per_state_summary.csv")
    display(state_summary)
    for regime in sources:
        if not clean[clean.regime == regime].empty:
            show_and_save(experiment.plot_confusions(run["confusion"], clean, regime),
                          f"{regime}_confusion_matrices")
            show_and_save(experiment.plot_per_state(run["per_state"], clean, regime),
                          f"{regime}_per_state_classification")

## True and fitted emission densities

Always show replicate 0 at the largest saved training size, rather than selecting
the best test outcome. The density window is ±5 true standard deviations around
each state's mean; it is a display window, not the KL integration domain.

In [ ]:
if report_ready:
    for regime, source in sources.items():
        for family in source.families:
            eligible = clean[(clean.regime == regime) & (clean.family == family)
                             & (clean.replicate == 0)]
            if not eligible.empty:
                show_and_save(experiment.plot_emissions(source, run_dir, family),
                              f"{regime}_{family}_fitted_emissions")

## Decoded state paths

Show the exact saved test observations, true states, and all four full-sequence
Viterbi paths. Black crosses indicate mistakes. These are retrospective decoded
paths; they do not imply causal online filtering.

In [ ]:
if report_ready:
    for regime, source in sources.items():
        eligible = clean[(clean.regime == regime) & (clean.family == "Trimodal")
                         & (clean.replicate == 0)]
        if not eligible.empty:
            for condition in ("clean", "noise_0.5"):
                show_and_save(experiment.plot_decoding(
                    source, run_dir, condition=condition, points=source.settings["n_test"]
                ), f"{regime}_{condition}_decoded_example")

## Noise robustness for mean and volatility regimes

No refitting occurs under test noise. The original schedule and seeds produce the
same perturbations for every emission model. Robustness summaries require all
four models to succeed on **all four conditions** for that replicate. Noise NLL
measures robustness under the clean fitted model, not clean-parameter recovery.

In [ ]:
if report_ready and not robust.empty:
    noise_metrics = ["f1_macro", "state_brier_score", "test_nll",
                     "absolute_switch_count_error"]
    robustness_summary = robust.groupby([
        "regime", "family", "n_train", "model", "condition"
    ])[noise_metrics + ["aligned_accuracy", "state_log_loss",
                       "mean_true_state_posterior", "switch_f1"]].agg(["mean", "std", "count"])
    robustness_summary.to_csv(run_dir / "noise_robustness_summary.csv")
    display(robustness_summary)
    noise_deltas = experiment.paired_differences(robust, noise_metrics)
    noise_deltas.to_csv(run_dir / "noise_paired_model_differences.csv", index=False)
    for regime in sources:
        if not robust[robust.regime == regime].empty:
            show_and_save(experiment.plot_metrics(robust, regime, noise_metrics, noise=True),
                          f"{regime}_noise_robustness")
    clean_scores = robust[robust.condition == "clean"]
    key = experiment.KEYS + ["model"]
    perturbation_deltas = robust.merge(clean_scores[key + noise_metrics],
                                      on=key, suffixes=("", "_clean"), validate="many_to_one")
    for metric in noise_metrics:
        perturbation_deltas[f"{metric}_noisy_minus_clean"] = (
            perturbation_deltas[metric] - perturbation_deltas[f"{metric}_clean"]
        )
    perturbation_deltas.to_csv(run_dir / "noisy_minus_clean.csv", index=False)

In [ ]:
if report_ready:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
    for ax, (regime, source) in zip(axes, sources.items()):
        data = source.data(0, "Gaussian")
        n_train = max(source.settings["train_sizes"])
        variants = source.variants(data["train"].iloc[:n_train], data["test"],
                                   0, "Gaussian", n_train)
        for condition, frame in variants.items():
            if condition != "clean":
                ax.plot(frame.time, frame.noise_std, label=condition)
        ax.set(title=f"{regime}: replicate 0, Gaussian data",
               xlabel="Test observation index", ylabel="Added-noise SD")
        ax.legend()
    show_and_save(fig, "measurement_noise_schedules")

## Fit reliability and computation

Inspect convergence and failed starts alongside test metrics. The final flow
budget inherits 10 starts and a 1,000-update cap, but a numerical M-step differs
in cost from closed-form Gaussian/GMM updates. Tuning cost is additional and is
recorded separately. Baseline timings are historical and may use different hardware.

With parallel workers, `total_fit_seconds` sums overlapping per-start durations;
it is not elapsed experiment time. Each stage's `execution_timing.json` records
wall-clock time for that invocation, including worker startup and any resumed
work. CPU throughput gains from the toy benchmark are not a guarantee for the
full experiment or multivariate flows.

In [ ]:
if report_ready:
    attempts = run["attempts"]
    fit_diagnostics = attempts.assign(failed=attempts.status.ne("ok")).groupby([
        "regime", "family", "n_train", "model"
    ]).agg(starts=("status", "size"), failures=("failed", "sum"),
           converged_starts=("converged", "sum"),
           mean_iterations=("iterations", "mean"),
           total_fit_seconds=("fit_seconds", "sum"),
           likelihood_decreases=("likelihood_decreases", "sum"))
    fit_diagnostics.to_csv(run_dir / "fit_diagnostics.csv")
    display(fit_diagnostics)
    history_fig, history_axes = plt.subplots(1, 2, figsize=(13, 4), layout="constrained")
    for ax, regime in zip(history_axes, sources):
        for model in experiment.MODELS:
            selected = attempts[(attempts.regime == regime)
                                & (attempts.family == "Trimodal")
                                & (attempts.replicate == 0)
                                & (attempts.model == model) & attempts.selected]
            if not selected.empty:
                history = selected.iloc[0]["history"]
                if isinstance(history, str):
                    import ast
                    history = ast.literal_eval(history)
                ax.plot(np.arange(len(history)), np.asarray(history) / 20000,
                        color=experiment.COLORS[model], linestyle=experiment.STYLES[model],
                        label=model)
        ax.set(title=f"{regime}: replicate 0, Trimodal", xlabel="Outer EM update",
               ylabel="Training log likelihood / observation")
    history_axes[0].legend(fontsize=8)
    show_and_save(history_fig, "selected_training_histories")

## Reference Gaussian KL floor

The Gaussian matching each true state's mean and variance minimizes
$\mathrm{KL}(p_{\rm true}\|q_{\rm Gaussian})$. This diagnoses unavoidable Gaussian
shape mismatch; it is not a lower bound on the KL achievable by the other models.

In [ ]:
if report_ready:
    floors = experiment.gaussian_floors(sources)
    floors.to_csv(run_dir / "gaussian_kl_floors.csv", index=False)
    display(floors)

## Interpretation and execution checklist

No winner or tuned configuration is reported until the experiments are run.
To execute later, enable `RUN_TUNING`, inspect validation results, then enable
`RUN_FINAL`. Use `REPORT_EXISTING=True` to rebuild graphs from completed outputs
without fitting again. Resume with identical inputs/settings and the same run name;
use a new name when settings change.

Before interpreting outcomes, verify baseline replay checks, four-model replicate
coverage, fitting failures/convergence, and quadrature warnings. Compare final
held-out test likelihood, state recovery, clean density recovery, and robustness
within each regime/family. A Student-t base changes tail behavior, but it does not
guarantee better likelihood or state recovery. Hyperparameters may differ between
bases, so this is a comparison of independently tuned models, not a controlled
base-only ablation. Set identical settings and skip independent selection only in
a separate, explicitly labeled ablation run.